# Genius API Sample Checks
Use this notebook to quickly search Genius and fetch lyrics for manual spot checks.

In [1]:
import os
import lyricsgenius

GENIUS_ACCESS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN", "")
if not GENIUS_ACCESS_TOKEN:
    raise ValueError("Set GENIUS_ACCESS_TOKEN in your environment or directly in this cell.")

genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    timeout=15,
    retries=3,
    remove_section_headers=True,
    skip_non_songs=True,
 )
genius.verbose = False

In [2]:
def fetch_lyrics_genius(client, title: str, artist: str) -> str:
    try:
        hit = client.search_song(title=title, artist=artist)
        if hit and hit.lyrics:
            return hit.lyrics.strip()
    except Exception as e:
        print(f"[warn] {title!r} by {artist!r}: {e}")
    return ""

In [87]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks → project root

titles_file_path = PROJECT_ROOT / "data/processed/titles/2026/03/05/titles.csv"
lyrics_file_path = PROJECT_ROOT / "data/processed/lyrics/2026/03/05/lyrics.csv"

titles_df = pd.read_csv(titles_file_path)

In [88]:
# STEP 1a: Search Title using spotify_uri from titles.csv
spotify_uri = "0ofHAoxe9vBkTCp2UQIavz"

matching_row = titles_df[titles_df["spotify_uri"] == spotify_uri]
song_title = matching_row["title"].iloc[0] if not matching_row.empty else None
song_artist = matching_row["artist"].iloc[0] if not matching_row.empty else None

print(f"Spotify URI: {spotify_uri}")
print(f"Title: {song_title}")
print(f"Artist: {song_artist}")

Spotify URI: 0ofHAoxe9vBkTCp2UQIavz
Title: Dreams
Artist: Fleetwood Mac


In [ ]:
# # STEP 1b: Search Lyrics using Title
# song_title = "爱怎么了"

In [83]:
# STEP 2: Fetch lyrics using Genius API
lyrics = fetch_lyrics_genius(genius, song_title, song_artist)

print("\nLyrics snippet:\n")
print((lyrics or "<no lyrics found>")[:1200])


Lyrics snippet:

(Mmm)

Now, here you go again
You say you want your freedom
Well, who am I to keep you down?
It's only right that you should
Play the way you feel it
But listen carefully to the sound
Of your loneliness

Like a heartbeat, (Heart) drives you mad (beat)
In the stillness (Still-) of remembering (-ness)
What you had (Lonely) and what you lost (Ooh, ooh)
And what you had (Ooh, ooh) and what you lost (Ooh)

Oh, thunder only happens when it's rainin'Players only love you when they're playing
Say, "Women, they will come and they will go"
When the rain washes you clean, you'll know
You'll know


Now, here I go again
I see the crystal visions
I keep my visions to myself
It's only me who wants to
Wrap around your dreams
And have you any dreams you'd like to sell?
Dreams of loneliness

Like a heartbeat, (Heart) drives you mad (beat)
In the stillness (Still-) of remembering (-ness)
What you had (Lonely) and what you lost (Ooh, ooh)
And what you had (Ooh, ooh) Ooh, what you lost (A

In [90]:
# STEP 2: Get spotify_uri from titles.csv by title and artist
# If artist is empty, search only by title

song_title_tmp = '' # for manual searching if we want to try a different title

if song_title_tmp:
    song_title = song_title_tmp

if song_artist != "":
    matching_row = titles_df[
        titles_df["title"].str.contains(song_title, case=False, na=False) &
        titles_df["artist"].str.contains(song_artist, case=False, na=False)
    ]
else:
    matching_row = titles_df[
        titles_df["title"].str.contains(song_title, case=False, na=False)
    ]

result = titles_df.loc[titles_df["title"] == song_title, ["title", "spotify_uri", "artist"]]
display(result)

song_uri = result["spotify_uri"].iloc[0] if not result.empty else None
song_artist = result["artist"].iloc[0] if not result.empty else None
song_title = result["title"].iloc[0] if not result.empty else None

print(f"\nSpotify URI of the song: {song_uri}")
print(f"Artist: {song_artist}")
print(f"Title: {song_title}")

,title,spotify_uri,artist
259,Dreams,0ofHAoxe9vBkTCp2UQIavz,Fleetwood Mac
625,Dreams,0ofHAoxe9vBkTCp2UQIavz,Fleetwood Mac



Spotify URI of the song: 0ofHAoxe9vBkTCp2UQIavz
Artist: Fleetwood Mac
Title: Dreams


In [85]:
# STEP 3: Update titles.csv with song_title and song_artist for the row with the matching spotify_uri (if found)
if song_uri:
    titles_df.loc[titles_df["spotify_uri"] == song_uri, ["title", "artist"]] = [song_title, song_artist]
    titles_df.to_csv(Path(titles_file_path), index=False)
    print(f"Updated titles.csv with title '{song_title}' and artist '{song_artist}' for spotify_uri '{song_uri}'.")
else:
    print(f"No matching spotify_uri found for title '{song_title}' and artist '{song_artist}'. No updates made to titles.csv.")

Updated titles.csv with title 'Dreams' and artist 'Fleetwood Mac' for spotify_uri '0ofHAoxe9vBkTCp2UQIavz'.


In [86]:
# STEP 4: Update lyrics.csv with lyrics for the matching spotify_uri
lyrics_df = pd.read_csv(Path(lyrics_file_path))
if song_uri and lyrics:
    lyrics_df.loc[lyrics_df["spotify_uri"] == song_uri, "lyrics"] = lyrics
    lyrics_df.to_csv(Path(lyrics_file_path), index=False)
    print(f"Updated lyrics.csv with lyrics for spotify_uri '{song_uri}'.")
else:
    print(f"No matching spotify_uri or lyrics found for title '{song_title}' and artist '{song_artist}'. No updates made to lyrics.csv.")

Updated lyrics.csv with lyrics for spotify_uri '0ofHAoxe9vBkTCp2UQIavz'.
